# UmojaHack Africa 2022 — Faulty Air Quality Sensor Detection

**Competition:** UmojaHack Africa 2022 Beginner Challenge (Zindi)
**Problem:** Binary classification — detect whether an air quality sensor is reporting an offset fault
**Dataset:** ~400,000 sensor readings from Uganda (PM2.5, temperature, humidity, datetime)
**Result:** F1 score of 0.9169 on validation set
**Model:** Gradient Boosting Classifier with 10-fold stratified cross-validation

---

## Problem context

Low-cost air quality sensors deployed across Uganda sometimes develop **offset faults** — systematic errors where readings are consistently shifted from the true value. Identifying which sensors are faulty allows public health agencies to correct or discard their readings before they influence policy decisions.

This is a **binary classification** task:
- `0` = Normal sensor
- `1` = Offset fault detected

The pipeline in this notebook mirrors the modular `.py` files in `src/` exactly:
- `src/preprocess.py` → Sections 1 and 2
- `src/features.py` → Section 3
- `src/train.py` → Sections 4 and 5

## 0. Imports

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import f1_score, classification_report, confusion_matrix

os.makedirs('../outputs', exist_ok=True)
print('[OK] Imports complete')

## 1. Load data
*(mirrors `preprocess.load_data()`)*

We parse the `Datetime` column immediately on load so all temporal operations downstream work without extra conversion.

In [ ]:
train = pd.read_csv('../data/train.csv', parse_dates=['Datetime'])
test  = pd.read_csv('../data/test.csv',  parse_dates=['Datetime'])
ss    = pd.read_csv('../data/SampleSubmission.csv')

print(f'Train shape: {train.shape}')
print(f'Test shape:  {test.shape}')
print(f'Columns: {list(train.columns)}')
train.head()

In [ ]:
# Class balance check
print('=== Target distribution ===')
print(train['Offset_fault'].value_counts())
print()
print(train['Offset_fault'].value_counts(normalize=True).round(4))

## 2. Preprocessing
*(mirrors `preprocess.impute_missing()` and `preprocess.log_transform()`)*

### 2.1 Missing value imputation

A small number of sensor readings are missing. We impute with **train medians** rather than means — medians are more robust to the right-skewed PM2.5 distributions we see in this dataset.

Critically, train medians are applied to the test set too — never compute statistics from test data to avoid leakage.

In [ ]:
# Inspect missing values
def missing_summary(df, name):
    total   = df.isnull().sum()
    pct     = (total / len(df) * 100).round(2)
    summary = pd.DataFrame({'Missing': total, 'Percent (%)': pct})
    print(f'\n=== Missing values: {name} ===')
    print(summary[summary['Missing'] > 0])

missing_summary(train, 'Train')
missing_summary(test,  'Test')

In [ ]:
# Impute with train medians — applied to both train and test
nan_cols = ['Sensor1_PM2.5', 'Sensor2_PM2.5', 'Temperature', 'Relative_Humidity']

for col in nan_cols:
    median_val = train[col].median()
    train[col] = train[col].fillna(median_val)
    test[col]  = test[col].fillna(median_val)  # train median only — no leakage

print('[OK] Missing values imputed')
print(train[nan_cols].isnull().sum())

### 2.2 Log transformation

PM2.5, temperature, and humidity readings are right-skewed. Log-transforming them compresses the tail, brings distributions closer to symmetric, and helps gradient boosting find better split points.

We use `log(x + 1)` to handle any zero values safely.

In [ ]:
log_cols = ['Relative_Humidity', 'Temperature', 'Sensor1_PM2.5', 'Sensor2_PM2.5']

# Visualise before and after for one column
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(train['Sensor1_PM2.5'], bins=50, color='steelblue', alpha=0.7)
axes[0].set_title('Sensor1_PM2.5 — before log transform')

for dataset in (train, test):
    dataset[log_cols] = np.log(dataset[log_cols] + 1)

axes[1].hist(train['Sensor1_PM2.5'], bins=50, color='tomato', alpha=0.7)
axes[1].set_title('Sensor1_PM2.5 — after log transform')
plt.tight_layout()
plt.savefig('../outputs/log_transform_comparison.png', dpi=120)
plt.show()
print(f'[OK] Log transformation applied to: {log_cols}')

## 3. Exploratory data analysis

With clean data in hand, we examine distributions and relationships to validate the feature engineering choices made in `src/features.py`.

In [ ]:
# PM2.5 distributions by fault status
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, col in zip(axes, ['Sensor1_PM2.5', 'Sensor2_PM2.5']):
    ax.hist(train[train['Offset_fault'] == 0][col], alpha=0.5,
            label='Normal', color='steelblue', bins=50)
    ax.hist(train[train['Offset_fault'] == 1][col], alpha=0.5,
            label='Faulty',  color='tomato',    bins=50)
    ax.set_title(f'{col} by fault status')
    ax.legend()
plt.tight_layout()
plt.savefig('../outputs/pm25_distributions.png', dpi=120)
plt.show()

In [ ]:
# Correlation heatmap on numeric features
numeric_cols = ['Sensor1_PM2.5', 'Sensor2_PM2.5', 'Temperature',
                'Relative_Humidity', 'Offset_fault']
plt.figure(figsize=(7, 5))
sns.heatmap(train[numeric_cols].corr(), annot=True, fmt='.2f',
            cmap='coolwarm', center=0)
plt.title('Feature correlation matrix')
plt.tight_layout()
plt.savefig('../outputs/correlation_matrix.png', dpi=120)
plt.show()

## 4. Feature engineering
*(mirrors `features.build_features()`)*

Raw sensor readings alone are insufficient. We create features that capture:

| Feature group | Rationale |
|---|---|
| Datetime decomposition | Temporal patterns — time-of-day, seasonality, maintenance cycles |
| PM2.5 bins | Threshold effects in sensor fault behaviour |
| S1_S2 delta | Inter-sensor divergence — the strongest individual fault signal |
| IsWeekend | Maintenance cycle proxy |
| IsHot | Temperature stress indicator |

In [ ]:
def build_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Datetime decomposition
    df['Datetime_day']          = df['Datetime'].dt.day
    df['Datetime_month']        = df['Datetime'].dt.month
    df['Datetime_year']         = df['Datetime'].dt.year
    df['Datetime_hour']         = df['Datetime'].dt.hour
    df['Datetime_minute']       = df['Datetime'].dt.minute
    df['Datetime_seconds']      = df['Datetime'].dt.second
    df['Datetime_day_of_year']  = df['Datetime'].dt.dayofyear
    df['Datetime_day_of_week']  = df['Datetime'].dt.dayofweek
    df['Datetime_week_of_year'] = df['Datetime'].dt.isocalendar().week.astype(int)
    df.drop('Datetime', axis=1, inplace=True)

    # PM2.5 bin features
    bins   = [-np.inf, 3.0, 3.5, 4.0, np.inf]
    labels = [1, 2, 3, 4]
    df['Sensor1_bins'] = pd.cut(df['Sensor1_PM2.5'], bins=bins, labels=labels)
    df['Sensor2_bins'] = pd.cut(df['Sensor2_PM2.5'], bins=bins, labels=labels)

    # Inter-sensor delta — key fault signal
    df['S1_S2'] = df['Sensor1_PM2.5'] - df['Sensor2_PM2.5']

    # Behavioural flags
    df['IsWeekend'] = (df['Datetime_day_of_week'] >= 5).astype(int)
    df['IsHot']     = (df['Temperature'] > df['Temperature'].mean()).astype(int)

    if 'ID' in df.columns:
        df.drop('ID', axis=1, inplace=True)

    return df.dropna()

train = build_features(train)
test  = build_features(test)

print(f'[OK] Train shape after feature engineering: {train.shape}')
print(f'[OK] Test shape after feature engineering:  {test.shape}')
print(f'[OK] Features: {[c for c in train.columns if c != "Offset_fault"]}')

## 5. Model training and cross-validation
*(mirrors `train.py` training loop)*

We use **10-fold Stratified Shuffle Split** cross-validation. Stratification ensures each fold preserves the class ratio from the full dataset — important for a potentially imbalanced binary target.

For each fold we:
1. Train a `GradientBoostingClassifier` on the training split
2. Predict on the validation split and record F1 score
3. Accumulate test predictions (averaged across folds for the final submission)
4. Track the best model by validation F1 for saving

In [ ]:
X = train.drop('Offset_fault', axis=1)
y = train['Offset_fault']

folds            = StratifiedShuffleSplit(n_splits=10, random_state=2021)
oof_preds        = np.zeros(len(X))
test_predictions = np.zeros(len(test))
fold_scores      = []
best_clf         = None
best_fold_score  = 0.0

print('[TRAINING] 10-fold Stratified Shuffle Split')
print('-' * 45)

for fold, (trn_idx, val_idx) in enumerate(folds.split(X, y)):
    X_trn, y_trn = X.iloc[trn_idx], y.iloc[trn_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    clf = GradientBoostingClassifier(random_state=42)
    clf.fit(X_trn, y_trn)

    val_preds          = clf.predict(X_val)
    val_f1             = f1_score(y_val, val_preds)
    fold_scores.append(val_f1)
    oof_preds[val_idx] = val_preds
    test_predictions  += clf.predict(test) / folds.n_splits

    if val_f1 > best_fold_score:
        best_fold_score = val_f1
        best_clf        = clf

    print(f'Fold {fold+1:02d} | Val F1: {val_f1:.4f}')

mean_f1 = float(np.mean(fold_scores))
print(f'\n[OK] Mean CV F1:  {mean_f1:.4f} (+/- {np.std(fold_scores):.4f})')
print(f'[OK] Best fold F1: {best_fold_score:.4f}')

## 6. Evaluation

In [ ]:
# Out-of-fold classification report
print('=== Out-of-fold Classification Report ===')
print(classification_report(y, oof_preds, target_names=['Normal', 'Faulty']))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y, oof_preds)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'Faulty'],
            yticklabels=['Normal', 'Faulty'])
plt.title('Out-of-fold Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('../outputs/confusion_matrix.png', dpi=120)
plt.show()
print('[OK] confusion_matrix.png saved')

In [ ]:
# Feature importance — top 15
importances = pd.Series(best_clf.feature_importances_, index=X.columns)
importances = importances.sort_values(ascending=True).tail(15)

plt.figure(figsize=(8, 6))
importances.plot(kind='barh', color='steelblue')
plt.title('Top 15 Feature Importances — GradientBoostingClassifier')
plt.xlabel('Importance')
plt.tight_layout()
plt.savefig('../outputs/feature_importance.png', dpi=120)
plt.show()
print('[OK] feature_importance.png saved')

## 7. Save model artifact and generate submission

In [ ]:
# Save model
model_artifact = {
    'model':         best_clf,
    'feature_names': list(X.columns),
    'mean_cv_f1':    mean_f1,
    'best_fold_f1':  best_fold_score,
}
model_path = '../outputs/gradient_boosting_model.pkl'
with open(model_path, 'wb') as f:
    pickle.dump(model_artifact, f)

print(f'[OK] Model saved to {model_path}')
print(f'[OK] Features: {len(X.columns)}')
print(f'[OK] Mean CV F1: {mean_f1:.4f}')

In [ ]:
# Generate submission file
ss['Offset_fault'] = [1 if x >= 0.5 else 0 for x in test_predictions]
ss.to_csv('../outputs/submission.csv', index=False)

print('[OK] Submission saved to ../outputs/submission.csv')
print(ss['Offset_fault'].value_counts())